# Gravando os passes em Parquet

## Task 4: gravar os passes por partida e por Copa em Parquet

## O problema

Gravar os passes em dois níveis de detalhe, **por partida** e **por Copa**, no formato do feed: **Parquet**.

In [ ]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-parquet-passes")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"
EVENTOS = "../../data/events"
OUT = "../../data/out"

spark

A célula abaixo já vem pronta: o código dos vídeos 4.1 a 4.3 (lances, partidas, ano e passes por partida e por Copa).

In [ ]:
# vídeo 4.1: os lances e as partidas
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

eventos = spark.read.parquet(EVENTOS).withColumnRenamed("MatchID", "match_id")

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)

# vídeo 4.2: o ano em cada lance
lances = eventos.join(partidas.select("match_id", "ano"), on="match_id")

# vídeo 4.3: os passes por partida e por Copa
passes_por_partida = (
    lances
    .filter(F.col("ano").isin(2018, 2022))
    .filter(F.col("event_type") == "Pass")
    .groupBy("ano", "match_id", "team")
    .agg(F.count("*").alias("passes"))
)

passes_por_copa = (
    passes_por_partida
    .groupBy("ano", F.col("team").alias("selecao"))
    .agg(
        F.sum("passes").alias("passes"),
        F.count("*").alias("partidas"),
        F.round(F.avg("passes"), 1).alias("media_por_partida"),
    )
)

## O que é o Parquet

O Parquet guarda os valores de **cada coluna juntos**, em binário e comprimidos (**colunar**), com o schema junto.

| No plano do 4.1 | Por causa do Parquet |
|---|---|
| `ReadSchema` | o leitor pula as colunas que não foram pedidas |
| `PushedFilters` | o leitor pula blocos inteiros, pelo menor e maior valor de cada coluna |
| `ColumnarToRow` | o dado sai do arquivo em lotes de colunas |
| nenhum `schema` na leitura | o schema vem dentro do arquivo |

## Gravando em Parquet

`mode("overwrite")`: se a pasta já existe, apaga e grava de novo.

`!` manda o comando para o terminal. O Spark grava **uma pasta**:

- `part-...parquet`: **um arquivo para cada partição** (aqui, um só: o AQE juntou as partições);
- `_SUCCESS`: criado só quando o job inteiro termina.

## Duas granularidades

| Saída | Cada linha é |
|---|---|
| `passes_por_partida` | uma seleção numa partida |
| `passes_por_copa` | uma seleção numa Copa |

## O que a gravação fez: a Spark UI

Spark UI (http://localhost:4040), aba **SQL / DataFrame**, a segunda consulta `parquet at`, a de `passes_por_copa`:

| Caixa | Métrica | O que ela mostra |
|---|---|---|
| `Scan parquet` | number of output rows | quantos lances a leitura trouxe |
| `BroadcastExchange` | number of output rows | quantas partidas foram copiadas para os executors |
| `Exchange` (o primeiro, depois do join) | shuffle records written | quantas linhas atravessaram o shuffle |
| `AQEShuffleRead` | number of partitions | em quantas partições o AQE juntou as 200 |
| `Execute InsertIntoHadoopFsRelationCommand` | number of written files | quantos arquivos a gravação criou |

As duas consultas começam no `Scan parquet` do feed: cada ação refaz o caminho inteiro (lazy evaluation, aula 2).

## Demanda da aula 4 atendida

| Etapa | Entrega |
|---|---|
| Task 1 | os gols do feed conferidos com os placares |
| Task 2 | `lances`, com o ano da Copa em cada lance |
| Task 3 | `passes_por_copa`, com passes, partidas e média |
| Task 4 | os passes por partida e por Copa em Parquet, e o job conferido na Spark UI |

## Para consultar depois

- **Cache:** `df.cache()` guarda o resultado de um DataFrame na memória da primeira vez que uma ação o calcula; as
  ações seguintes leem dali. Vale quando um resultado **caro** é usado **várias vezes**. Para um resultado pequeno, pode
  até atrasar. Confira na aba **Storage** da Spark UI e libere com `df.unpersist()`.
- **Compressão do Parquet:** o padrão é o `snappy`, rápido para comprimir e ler. Com `.option("compression", "zstd")`,
  os arquivos ficam menores, em troca de mais processamento.
- **Outros formatos colunares:** o ORC segue a mesma ideia do Parquet.
- **Aba Stages:** mostra cada stage com o tempo de cada task (**Min**, **Median**, **Max**). Uma task muito mais lenta
  que as outras costuma ser **data skew**: uma chave com muito mais linhas que as outras.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()